# Step 3 - Classification training
Pre-trained GoogLeNet family model (InceptionV3 in Keras) with a new classification layer.

## Import libraries

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers

## Hyperparameters

In [ ]:
SPLIT_DIR = Path("../data/classification/processed")
MODEL_DIR = Path("../models/classification")

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 0.001

## Load the images by batches
`image_size` resizes every image to 224x224. Only the train set is shuffled.

In [ ]:
def load_dataset(folder, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        folder,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=42,
    )

In [ ]:
train_ds = load_dataset(SPLIT_DIR / "train", shuffle=True)
val_ds = load_dataset(SPLIT_DIR / "val", shuffle=False)

class_names = train_ds.class_names
print(class_names)

## Preprocessing
Convert to float and normalize the pixels from [0, 255] to [-1, 1], like InceptionV3 expects.

In [ ]:
def preprocess(images, labels):
    images = tf.cast(images, tf.float32)
    images = tf.keras.applications.inception_v3.preprocess_input(images)
    return images, labels

In [ ]:
train_ds = train_ds.map(preprocess).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(preprocess).prefetch(tf.data.AUTOTUNE)

In [ ]:
images, labels = next(iter(train_ds))
print("Batch shape:", images.shape)
print("Min:", images.numpy().min(), "Max:", images.numpy().max())